In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import pickle

from dPCA import dPCA

from imports import *
from config import dir_config, main_config, ephys_config
from src.utils import pmf_utils, plot_utils, ephys_utils
import os
os.environ["PYDEVD_WARN_SLOW_RESOLVE_TIMEOUT"] = "2.0"
import copy

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_venn import venn3,venn2

from scipy.spatial.distance import cosine

In [ ]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [ ]:
align_labels = ephys_config["alignment_settings_GP"]

marginalization = {
    's': 'Stimulus',
    'c': 'Choice',
    't': 'Time'
}
n_components = 3

In [ ]:
toRF_colors = ["#96D6EC", "#6FC3EB", "#5289C6", "#4469B1"]
awayRF_colors = ["#F2A448", "#EF8D41", "#EC6A50", "#AC3626"]


plot_params = {
    "color": {0: {0:toRF_colors[0], 1:awayRF_colors[0]}, 1: {0:toRF_colors[1], 1:awayRF_colors[1]}, 2: {0:toRF_colors[2], 1:awayRF_colors[2]}, 3: {0:toRF_colors[3], 1:awayRF_colors[3]}},
    "lw": {0: 2, 1: 1},
    "linestyle": {0: "-", 1: "--"},
    "opacity": {0: 1, 1: 0.3},
}

# plot_params = {
#     "color": {0: toRF_colors[0], 1: toRF_colors[1], 2: toRF_colors[2], 3: toRF_colors[3]},
#     "lw": {0: 2, 1: 1},
#     "linestyle": {0: "-", 1: "--"},
#     "opacity": {0: 1, 1: 0.3},
# }

## Utils

In [ ]:
def plot_component(ax, time, data, title=None, plot_mean=False, sig_mask=None, axes_to_loop=None, color=None, alpha=1, linestyle=None, lw=2,ylim=None):
    # Default axes to loop over if not provided (all axes: b, s, and c)
    if axes_to_loop is None:
        axes_to_loop = list(marginalization.keys())[:-1]  # Loop over all axes: bias (b), stimulus (s), and choice (c)
    axis_indices = {marg: marg_idx for marg_idx,marg in enumerate(list(marginalization.keys())[:-1])}
    axes_not_to_loop = [axis_indices[axis] for axis in axis_indices if axis not in axes_to_loop]

    print(f"axes_to_loop: {axes_to_loop}, axes_not_to_loop: {axes_not_to_loop}")
    # Loop over the specified axes (s, c)

    for s in [0, 1,2,3] if 's' in axes_to_loop else [0]:
        for c in range(2) if 'c' in axes_to_loop else [0]:
            # If we are not plotting the mean, plot the raw data
            if not plot_mean:
                ax.plot(
                    time,
                    np.squeeze(data[s, c, :]),  # Raw data
                    color=plot_params["color"][s][c] if color is None else color,
                    linewidth=lw,
                    alpha=alpha,
                    linestyle=plot_params["linestyle"][0],# if linestyle is None else linestyle,
                )
            else:
                axes = {'s': s, 'c': c}
                indices = [axes[axis] if axis in axes_to_loop else slice(None) for axis in axis_indices]
                index_tuple = tuple(indices) + (slice(None),)
                # Now use the tuple to index the array
                mean_data = np.mean(np.squeeze(data[index_tuple]), axis=tuple(range(len(axes_not_to_loop))))
                shade_colors = ['#b3b3b3', '#7f7f7f', '#4d4d4d', '#000000']

                if color is None:
                    if axes_to_loop == ['s']:
                        color_plot = shade_colors[s]
                    elif axes_to_loop == ['c']:
                        color_plot = plot_params["color"][3][c]
                else:
                    color_plot = color
                ax.plot(
                    time,
                    mean_data,  # Plot the mean over the non-looped axes
                    color=color_plot,
                    linewidth=lw,
                    alpha=alpha,
                    linestyle=plot_params["linestyle"][0]# if linestyle is None else linestyle,
                )

            lower_ylim = np.nanmin(data)-((np.nanmax(data) - np.nanmin(data))*0.05)-0.05
            upper_ylim = np.nanmax(data) + ((np.nanmax(data) - np.nanmin(data))*0.05)+0.05
            if sig_mask is not None:
                ax.imshow(sig_mask, extent=[time[0], time[-1], lower_ylim, np.amin(data)-0.05], aspect='auto', cmap='gray_r', vmin=0, vmax=1)
            ax.set_title(title)
            # ax.set_ylim(ylim) if ylim else ax.set_ylim(lower_ylim, upper_ylim)
            ax.axvline(0, color="black", linestyle="--", linewidth=1)

In [ ]:
def plot_dPCA_projection_sig(plot_mean, plot_sig_mask, ephys_config, dpca_results):
    alignment_dict = ephys_config["alignment_settings_GP"]
    for alignment in alignment_dict.keys():
        # Get transformed data
        Z = dpca_results[alignment]["transformed_data"]
        sig_mask = significance_masks[alignment] if plot_sig_mask else None

        # Align the duration based on the alignment type
        aligned_duration = np.arange(alignment_dict[alignment]["start_time_ms"], alignment_dict[alignment]["end_time_ms"] + 1)
        time = aligned_duration[-np.arange(Z['t'].shape[-1]) - 1][::-1] if alignment == "response" else aligned_duration[np.arange(Z['t'].shape[-1])]
        # Set step size based on alignment type
        step = 50 if alignment == "response" else 100

        fig, axes = plt.subplots(3, 1,figsize=(6, 12),sharex=True)

        # Loop through the principal components (PCs)
        PC_num = 0  # Plot only the first principal component
        if plot_mean:
            plot_component(axes[0], time, Z['t'][PC_num,:], f"time PC {PC_num+1}")
            plot_component(axes[1], time, Z['s'][PC_num, :], f"stimulus PC {PC_num+1}", plot_mean=True, sig_mask=sig_mask['s'][PC_num][None,:] if plot_sig_mask else None,
                           axes_to_loop=['s'], alpha=plot_params["opacity"][0], linestyle=plot_params["linestyle"][0],lw=5)
            plot_component(axes[2], time, Z['c'][PC_num, :], f"choice PC {PC_num+1}", plot_mean=True, sig_mask=sig_mask['c'][PC_num][None,:] if plot_sig_mask else None,
                           axes_to_loop=['c'], alpha=plot_params["opacity"][0],lw=5)
            # plot_component(axes[4], time, Z['bs'][PC_num, :], f"stimulus-bias interaction PC {PC_num+1}", sig_mask=sig_mask['bs'][PC_num][None,:] if plot_sig_mask else None,
            #                plot_mean=True, axes_to_loop=['b','s'],linestyle=plot_params["linestyle"][0])


        plot_component(axes[0], time, Z['t'][PC_num,:], f"time PC {PC_num+1}")
        plot_component(axes[1], time, Z['s'][PC_num,:], f"stimulus PC {PC_num+1}", alpha=0.3, sig_mask=sig_mask['s'][PC_num][None,:] if plot_sig_mask else None)
        plot_component(axes[2], time, Z['c'][PC_num,:], f"choice PC {PC_num+1}", alpha=0.3, sig_mask=sig_mask['c'][PC_num][None,:] if plot_sig_mask else None)
            # plot_component(axes[4], time, Z['bs'][PC_num,:], f"stimulus-bias interaction PC {PC_num+1}", sig_mask=sig_mask['bs'][PC_num][None,:] if plot_sig_mask else None)
        for ax in axes:
            # ax.set_title('')
            ax.set_xticks([])
            # ax.set_yticks([])
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)
            ax.spines['bottom'].set_visible(False)
            ax.tick_params(axis='both', which='major', labelsize=15)
        # axes[2].set_xticks(np.arange(time[0], time[-1]+1, step))

        shade_colors = ['#b3b3b3', '#7f7f7f', '#4d4d4d', '#000000']
        # Add the legend for different lines
        handles = [
            plt.Line2D([0], [0], color=shade_colors[3], label="50% coh"),
            plt.Line2D([0], [0], color=shade_colors[2], label="20% coh"),
            plt.Line2D([0], [0], color=shade_colors[1], label="6% coh"),
            plt.Line2D([0], [0], color=shade_colors[0], label="0% coh"),
            plt.Line2D([0], [0], color=plot_params["color"][3][0], label="toRF choice"),
            plt.Line2D([0], [0], color=plot_params["color"][3][1], label="awayRF choice"),
        ]
        fig.legend(handles=handles, bbox_to_anchor=(1.25, 1), loc='upper right')
        fig.suptitle(f"Aligned to {alignment.capitalize().replace('_', ' ')}")

        # Add time scale bar to the bottom axis (axes[2])
        scalebar_length = 100  # e.g., 200 ms
        xlim = axes[2].get_xlim()
        ylim = axes[2].get_ylim()

        # Position: bottom-right with padding
        x_start = xlim[1] - scalebar_length -150 # 100 ms padding from right edge
        y_start = ylim[0] + 0.1 * (ylim[1] - ylim[0])  # slight padding from bottom

        # Draw horizontal bar
        axes[2].hlines(y_start, x_start, x_start + scalebar_length, color='k', linewidth=5)

        # Add label centered below the bar
        axes[2].text(x_start + scalebar_length / 2,
                     y_start - 0.05 * (ylim[1] - ylim[0]),
                     f"{scalebar_length} ms", ha='center', va='top', fontsize=30)
        plt.tight_layout()
        plt.show()


In [ ]:
def classification(class_means, test):
    Q = class_means.shape[0]
    T = class_means.shape[1]
    performance = np.zeros(T)
    distance = 0.

    # classify every data point in test according to nearest class mean
    for t in range(T):
        if np.any(np.isnan(test[:,t])): # if any test data is nan, skip this time point
            performance[t] = np.nan
        else:
            for p in range(Q):
                argmin = 0
                distance = abs(class_means[argmin,t] - test[p,t])

                # find closest class mean
                for q in range(1,Q):
                    if abs(class_means[q,t] - test[p,t]) < distance:
                        distance = abs(class_means[q,t] - test[p,t])
                        argmin = q

                # add 1 if class is correct
                if argmin == p:
                    performance[t] += 1.

        performance[t] /= Q

    return performance

def denoise_mask(mask, n_consecutive):
    subseq = 0
    N = mask.shape[0]

    for n in range(N):
        if mask[n] == 1:
            subseq += 1
        else:
            if subseq < n_consecutive:
                for k in range(n-subseq,n):
                    mask[k] = 0

    return mask

def flat2d(A):
        ''' Flattens all but the first axis of an ndarray, returns view. '''
        return A.reshape((A.shape[0],-1))

def dpca_transform(dpca, X):
    X = X - np.nanmean(X.reshape((X.shape[0],-1)),1).reshape((X.shape[0],) + (len(X.shape)-1)*(1,))
    total_variance = np.sum((X - np.nanmean(X))**2)
    def marginal_variances(marginal):
        ''' Computes the relative variance explained of each component
            within a marginalization
        '''
        D, Xr = dpca.D[marginal], X.reshape((X.shape[0],-1))
        return [np.sum(np.dot(D[:,k], Xr)**2) / total_variance for k in range(D.shape[1])]

    X_transformed = {}
    dpca.explained_variance_ratio_ = {}
    for key in list(dpca.marginalizations.keys()):
        X_transformed[key] = np.dot(dpca.D[key].T, X.reshape((X.shape[0],-1))).reshape((dpca.D[key].shape[1],) + X.shape[1:])
        dpca.explained_variance_ratio_[key] = marginal_variances(key)
    return dpca, X_transformed

def dpca_train_test_split(dpca,X,trialX,N_samples=None,sample_ax=0):
    protect = dpca.protect
    n_unprotect = len(X.shape) - len(protect) if protect is not None else len(X.shape)
    n_protect   = len(protect) if protect is not None else 0

    if sample_ax != 0:
        raise NotImplemented('The sample axis needs to come first.')

    # test if all protected axes lie at the end
    protected = dpca._check_protected(trialX,protect)

    # reorder matrix to protect certain axis (for speedup)
    if ~protected:
        # turn crossval_protect into index listX
        axes = [dpca.labels.index(ax) + 2 for ax in protect]

        # reorder matrix
        trialX = dpca._roll_back(trialX,axes)
        X = np.squeeze(dpca._roll_back(X[None,...],axes))

    # compute number of samples in each condition
    if N_samples is None:
        N_samples = dpca._get_n_samples(trialX,protect=dpca.protect)

    # get random indices
    idx = (np.random.rand(*N_samples.shape)*N_samples).astype(int)

    # select values
    blindX = np.empty(trialX.shape[1:])

    # iterate over multi_index
    it = np.nditer(np.empty(N_samples.shape), flags=['multi_index'])

    while not it.finished:
        blindX[it.multi_index + (np.s_[:],)*n_protect] = trialX[(idx[it.multi_index],) + it.multi_index + (np.s_[:],)*n_protect]
        it.iternext()

    # compute trainX
    trainX = (X * (N_samples / (N_samples - 1))[(np.s_[:],) * n_unprotect + (None,) * n_protect]
          - np.where(np.isnan(blindX), X, blindX) / (N_samples - 1)[(np.s_[:],) * n_unprotect + (None,) * n_protect])

    # trainX = (X*(N_samples/(N_samples-1))[(np.s_[:],)*n_unprotect + (None,)*n_protect] - blindX/(N_samples-1)[(np.s_[:],)*n_unprotect + (None,)*n_protect])

    # inverse rolled axis in blindX
    if ~protected:
        blindX = dpca._roll_back(blindX[...,None],axes,invert=True)[...,0]
        trainX = dpca._roll_back(trainX[...,None],axes,invert=True)[...,0]

    # remean datasets (both equally)
    trainX -= np.nanmean(flat2d(trainX),1)[(np.s_[:],) + (None,)*(len(X.shape)-1)]
    blindX -= np.nanmean(flat2d(blindX),1)[(np.s_[:],) + (None,)*(len(X.shape)-1)]

    return trainX, blindX

def compute_mean_score(dpca, X, trialX, n_splits, keys, axis=None):
    K = X.shape[-1]

    if type(dpca.n_components) == int:
        scores = {key : np.empty((dpca.n_components, n_splits, K)) for key in keys}
    else:
        scores = {key : np.empty((dpca.n_components[key], n_splits, K)) for key in keys}

    for shuffle in range(n_splits):
        print('.', end=' ')

        # do train-validation split
        trainX, validX = dpca_train_test_split(dpca,X,trialX)
        # fit a dPCA model to training data & transform validation data
        dpca.fit(trainX)
        dpca, trainZ = dpca_transform(dpca, trainX)
        dpca, validZ = dpca_transform(dpca, validX)

        # reshape data to match Cython input
        for key in keys:
            ncomps = dpca.n_components if type(dpca.n_components) == int else dpca.n_components[key]

            # mean over all axis not in key
            axset = dpca.marginalizations[key]
            axset = axset if type(axset) == set else set.union(*axset)
            if axis is None:
                axes = set(range(len(X.shape)-1)) - axset
            else:
                axes = set(range(len(X.shape)-1)) - axset - {len(X.shape)-2}
            for ax in list(axes)[::-1]:
                trainZ[key] = np.nanmean(trainZ[key],axis=ax+1)
                validZ[key] = np.nanmean(validZ[key],axis=ax+1)

            # reshape
            if (len(X.shape)-2 in axset) or (axis):
                trainZ[key] = trainZ[key].reshape((ncomps,-1,K))
                validZ[key] = validZ[key].reshape((ncomps,-1,K))
            else:
                trainZ[key] = trainZ[key].reshape((ncomps,-1,1))
                validZ[key] = validZ[key].reshape((ncomps,-1,1))

        # compute classification score
        for key in keys:
            ncomps = dpca.n_components if type(dpca.n_components) == int else dpca.n_components[key]
            for comp in range(ncomps):
                scores[key][comp, shuffle] = classification(trainZ[key][comp],validZ[key][comp])

    for key in keys:
        scores[key] = np.nanmean(scores[key], axis=1)

    return scores

def dpca_significance_analysis(dpca, X, trialX, n_shuffles=100, n_splits=100, n_consecutive=10, axis=None, full=False):

    if dpca.opt_regularizer_flag:
        print("Regularization not optimized yet; start optimization now.")
        dpca._optimize_regularization(X,trialX)

    keys = list(dpca.marginalizations.keys())
    keys.remove(dpca.labels[-1])

    # shuffling is in-place, so we need to copy the data
    trialX_shuffled = trialX.copy()

    # compute score of original data
    print("Compute score of data: ", end=' ')
    true_score = compute_mean_score(dpca, X, trialX, n_splits, keys, axis=axis)
    print("Finished.")

    # data collection
    scores = {key : [] for key in keys}
    shuffled_transformed_data = [{} for _ in range(n_shuffles)]

    # iterate over shuffles
    for it in range(n_shuffles):
        print("\rCompute score of shuffled data: ", str(it), "/", str(n_shuffles), end=' ')
        trialX_shuffled = dpca.shuffle_labels(trialX_shuffled)
        # mean trial-by-trial data
        X_shuffled = np.nanmean(trialX_shuffled,axis=0)
        # remove any timepoints with nan in X
        no_any_nan_mask = ~np.any(np.isnan(X_shuffled), axis=(0, 1, 2, 3))
        X_shuffled = X_shuffled[...,no_any_nan_mask]
        trialX_shuffled = trialX_shuffled[...,no_any_nan_mask]
        dpca, shuffled_transformed_data[it] = dpca_transform(dpca,X_shuffled)

        score = compute_mean_score(dpca,X_shuffled,trialX_shuffled,n_splits,keys,axis=axis)
        for key in keys:
            scores[key].append(score[key])

    # binary mask, if data score is above maximum shuffled score make true
    masks = {key: np.full(true_score[key].shape, False) for key in keys} # default is False
    for key in keys:
        min_length = min(shuffle_score.shape[1] for shuffle_score in scores[key]) # find minimum length of shuffled scores
        # maxscore = np.amax(np.dstack([shuffle_score[:,:min_length] for shuffle_score in scores[key]]),-1)
        quantile_score = np.nanquantile(np.dstack([shuffle_score[:,:min_length] for shuffle_score in scores[key]]), 0.95, axis=-1)
        masks[key][:,:min_length] = true_score[key][:,:min_length] > quantile_score

    if n_consecutive > 1:
        for key in keys:
            mask = masks[key]
            for k in range(mask.shape[0]):
                masks[key][k,:] = denoise_mask(mask[k].astype(np.int32),n_consecutive)

    if full:
        return masks, true_score, scores, shuffled_transformed_data
    else:
        return masks

def dpca_shuffle(dpca, trialX, n_shuffles = 10):
    shuffled_Z = {}
    for n_shuffle in range(n_shuffles):
        # shuffle labels
        trialX = dpca.shuffle_labels(trialX)
        # mean trial-by-trial data
        X = np.nanmean(trialX,axis=0)
        # remove any timepoints with nan in X
        no_any_nan_mask = ~np.any(np.isnan(X), axis=(0, 1, 2, 3))
        X = X[...,no_any_nan_mask]
        trialX = trialX[...,no_any_nan_mask]
        dpca.fit(X,trialX)
        _, shuffled_Z[n_shuffle] = dpca_transform(dpca, X)
    min_timepoints = np.min([shuffled_Z[n_shuffle]['t'].shape[4] for n_shuffle in range(n_shuffles)])
    mean_shuffled_Z = {key:[] for key in dpca.explained_variance_ratio_}
    for key in dpca.explained_variance_ratio_:
        mean_shuffled_Z[key] = np.mean(np.array([shuffled_Z[n_shuffle][key][...,0:min_timepoints] for n_shuffle in range(n_shuffles)]), axis=0)

    return shuffled_Z, mean_shuffled_Z

## Load data

In [ ]:
session_to_exclude = ["210210_GP_JP","241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, 'sessions_metadata.csv'))
session_metadata = session_metadata[~np.isin(session_metadata["session_id"],session_to_exclude)]

neuron_metadata = pd.read_csv(Path(processed_dir, 'neuron_metadata.csv'))
neuron_metadata = neuron_metadata[~np.isin(neuron_metadata["session_id"],session_to_exclude)]
neuron_to_exclude = [4, 16, 33, 54, 80, 87, 89, 103, 135, 153, 159, 162]
neuron_metadata = neuron_metadata[~np.isin(neuron_metadata["neuron_id"],neuron_to_exclude)].reset_index(drop=True)

with open(Path(processed_dir, f'dpca_data_and_projection_equal_only_all_sessions_1prct_outlier.pkl'), 'rb') as f:
    dpca_data_and_projection = pickle.load(f)
dpca_projection = dpca_data_and_projection["projection"]

In [ ]:
# project back to cropped average data
average_data = dpca_data_and_projection["data"]["average"]
for alignment in ephys_config["alignment_settings_GP"]:
    dpca = dpca_projection[alignment]["model"]
    _, dpca_projection[alignment]["transformed_data"] = dpca_transform(dpca, average_data[alignment])

### Significance Mask

In [ ]:

# significance_masks, true_score, scores, shuffled_data = {"toRF_prior": {}, "awayRF_prior": {}},{"toRF_prior": {}, "awayRF_prior": {}},{"toRF_prior": {}, "awayRF_prior": {}}, {"toRF_prior": {}, "awayRF_prior": {}}
# for prior in ["toRF_prior","awayRF_prior"]:
#     print(f"Prior: {prior.split('_')[0]}")
#     significance_masks[prior] = {}
#     true_score[prior] = {}
#     scores[prior] = {}
#     shuffled_data[prior] = {}
#     for alignment in ephys_config["alignment_settings_GP"]:

#         model = dpca_projection[alignment][prior]["model"]
#         dPCA_averaged_data = dpca_data_and_projection["data"]["average"]['toRF_prior'][alignment]
#         dPCA_trial_wise_data = dpca_data_and_projection["data"]["trial_wise"]['toRF_prior'][alignment]


#         significance_masks[prior][alignment], true_score[prior][alignment], scores[prior][alignment], shuffled_data[prior][alignment] = dpca_significance_analysis(
#             model,dPCA_averaged_data, dPCA_trial_wise_data, n_shuffles=100,n_splits=100, axis=True, full=True)

### plot projection

#### plot trajectories in choice PC-stimulus PC 2d figure

In [ ]:
fig,axs = plt.subplots(1,4,figsize = (20,5),sharex=True, sharey=True)
projection_epoch = 'response'
for idx, alignment in enumerate(ephys_config["alignment_settings_GP"]):
    ax = axs[idx]
    alignment_dict = ephys_config["alignment_settings_GP"]
    Z = dpca_projection[projection_epoch][alignment]
    aligned_duration = np.arange(alignment_dict[alignment]["start_time_ms"], alignment_dict[alignment]["end_time_ms"] + 1)
    PC_num = 0
    

    for s in [1,3]:
        for c in range(1):
            ax.plot(
                np.squeeze(Z['s'][0,:][s,c,:]),
                np.squeeze(Z['s'][1,:][s,c,:]),  # Raw data
                color=plot_params["color"][s][c],
                linestyle=plot_params["linestyle"][0],
            )
            ax.scatter(
                np.squeeze(Z['s'][0,:][s,c,0]),
                np.squeeze(Z['s'][1,:][s,c,0]),
                color=plot_params["color"][s][c],
                marker='o',
                edgecolor='k',
                s=100
            )
            # last_nonnan = np.where(~np.isnan(Z['s'][PC_num,:][s,c,:]))[0][-1]
            ax.scatter(
                np.squeeze(Z['s'][0,:][s,c,np.where(~np.isnan(Z['s'][PC_num,:][s,c,:]))[0][-1]]),
                np.squeeze(Z['s'][1,:][s,c,np.where(~np.isnan(Z['s'][PC_num,:][s,c,:]))[0][-1]]),
                color=plot_params["color"][s][c],
                marker='s',
                edgecolor='k',
                s=100
            )
    ax.set_title(f'{alignment} epoch')
    ax.set_xlabel('Stimulus PC 1 Projection')
    ax.set_xticks([])
axs[0].set_ylabel('Stimulus PC 2 Projection')
#plot legend
shade_colors = ['#b3b3b3', '#7f7f7f', '#4d4d4d', '#000000']
handles = [
            plt.Line2D([0], [0], color=shade_colors[3], label="50% coh"),
            plt.Line2D([0], [0], color=shade_colors[2], label="20% coh"),
            plt.Line2D([0], [0], color=shade_colors[1], label="6% coh"),
            plt.Line2D([0], [0], color=shade_colors[0], label="0% coh"),
            plt.Line2D([0], [0], color=plot_params["color"][3][0], label="toRF choice"),
            plt.Line2D([0], [0], color=plot_params["color"][3][1], label="awayRF choice"),
        ]
fig.legend(handles=handles, bbox_to_anchor=(1, 1), loc='upper right')
fig.suptitle('Response-aligned dPCA Projections')


In [ ]:
import plotly.graph_objects as go
import numpy as np

projection_epoch = 'response'

fig = go.Figure()

for alignment in ['cue']:  # replace with ephys_config["alignment_settings_GP"] if needed
    alignment_dict = ephys_config["alignment_settings_GP"]
    Z = dpca_projection[projection_epoch][alignment]
    aligned_duration = np.arange(alignment_dict[alignment]["start_time_ms"],
                                 alignment_dict[alignment]["end_time_ms"] + 1)
    
    for s in [1, 3]:
        for c in range(2):
            # extract PCs
            x = aligned_duration
            y = np.squeeze(Z['s'][0, s, c, :])
            z = np.squeeze(Z['s'][1, s, c, :])

            # remove NaNs
            valid = ~np.isnan(x) & ~np.isnan(y) & ~np.isnan(z)
            x, y, z = x[valid], y[valid], z[valid]

            # trajectory line
            fig.add_trace(go.Scatter3d(
                x=x, y=y, z=z,
                mode='lines',
                line=dict(color=plot_params["color"][s][c], width=4),
                name=f's={s}, c={c}'
            ))

            # start point
            fig.add_trace(go.Scatter3d(
                x=[x[0]], y=[y[0]], z=[z[0]],
                mode='markers',
                marker=dict(size=8, color=plot_params["color"][s][c], symbol='circle', line=dict(color='black', width=2)),
                showlegend=False
            ))

            # end point
            fig.add_trace(go.Scatter3d(
                x=[x[-1]], y=[y[-1]], z=[z[-1]],
                mode='markers',
                marker=dict(size=8, color=plot_params["color"][s][c], symbol='square', line=dict(color='black', width=2)),
                showlegend=False
            ))

# Update layout
fig.update_layout(
    title='Response-aligned dPCA 3D Projections',
    scene=dict(
        xaxis_title='Time',
        yaxis_title='Stimulus PC 1',
        zaxis_title='Stimulus PC 2'
    ),
    legend=dict(
        itemsizing='constant',
        x=1.05,
        y=1
    )
)

fig.show()


In [ ]:
baseline_projection = dpca_projection["baseline"]
visual_projection = dpca_projection["visual"]
cue_projection = dpca_projection["cue"]
response_projection = dpca_projection["response"]

In [ ]:
plot_dPCA_projection_sig(plot_mean=True, plot_sig_mask = False, ephys_config=ephys_config, dpca_results=dpca_projection)


### whole-trial projection

In [ ]:
def plot_component(ax, time, data, title=None, plot_mean=False, sig_mask=None, axes_to_loop=None, color=None, alpha=None, linestyle=None, ylim=None,lw=1):
    # Default axes to loop over if not provided (all axes: b, s, and c)
    if axes_to_loop is None:
        axes_to_loop = list(marginalization.keys())[:-1]  # Loop over all axes: bias (b), stimulus (s), and choice (c)
    axis_indices = {marg: marg_idx for marg_idx,marg in enumerate(list(marginalization.keys())[:-1])}
    axes_not_to_loop = [axis_indices[axis] for axis in axis_indices if axis not in axes_to_loop]


    # Loop over the specified axes (s, c)
    for s in [1,3] if 's' in axes_to_loop else [0]:
        for c in range(2) if 'c' in axes_to_loop else [0]:
            # If we are not plotting the mean, plot the raw data
            if not plot_mean:
                ax.plot(
                    time,
                    np.squeeze(data[s, c, :]),  # Raw data
                    color=plot_params["color"][s][c] if color is None else color,
                    # alpha=plot_params["opacity"][b] if alpha is None else alpha,
                    linestyle=plot_params["linestyle"][0] if linestyle is None else linestyle,
                    linewidth=lw,
                )
            else:
                axes = {'s': s, 'c': c}
                indices = [axes[axis] if axis in axes_to_loop else slice(None) for axis in axis_indices]
                index_tuple = tuple(indices) + (slice(None),)
                mean_data = np.mean(np.squeeze(data[index_tuple]), axis=tuple(range(len(axes_not_to_loop))))
                ax.plot(
                    time,
                    mean_data,  # Plot the mean over the non-looped axes
                    color=plot_params["color"][s][c] if color is None else color,
                    # alpha=plot_params["opacity"][b] if alpha is None else alpha,
                    linestyle=plot_params["linestyle"][0] if linestyle is None else linestyle,
                    linewidth=lw,
                )

            lower_ylim = np.nanmin(data)-((np.nanmax(data) - np.nanmin(data))*0.05)-0.05
            upper_ylim = np.nanmax(data) + ((np.nanmax(data) - np.nanmin(data))*0.05)+0.05
            if sig_mask is not None:
                ax.imshow(sig_mask, extent=[time[0], time[-1], lower_ylim, np.amin(data)-0.05], aspect='auto', cmap='gray_r', vmin=0, vmax=1)
            ax.set_title(title)
            ax.set_ylim(ylim) if ylim else ax.set_ylim(lower_ylim, upper_ylim)
            ax.axvline(0, color="black", linestyle="--", linewidth=1)

def plot_single_pc_projection(ax,dpca_axis, dpca_results, pc_component=0, ylim=None, legend=False, axes_to_loop=None, color=None,lw=1, title=None,flip=False):

    PC_num = pc_component
    alignment_dict = ephys_config["alignment_settings_GP"]
    alignment_fields = ["visual", "cue", "response"]

    # fig, ax = plt.subplots(len(alignment_fields),1, figsize = (5,5*len(alignment_fields)))
    for idx, alignment in enumerate(alignment_fields):
        # Get transformed data
        Z = dpca_results[alignment]
        sig_mask = None #significance_masks[prior][alignment] if plot_sig_mask else None

        # Align the duration based on the alignment type
        aligned_duration = np.arange(alignment_dict[alignment]["start_time_ms"], alignment_dict[alignment]["end_time_ms"] + 1)
        time = aligned_duration[-np.arange(Z['t'].shape[-1]) - 1][::-1] if alignment == "response" else aligned_duration[np.arange(Z['t'].shape[-1])]
        # Set step size based on alignment type
        step = 50 if alignment == "response" else 100

        x = time
        if dpca_axis == 'time':
            y, label,axes_to_loop = Z['t'][PC_num,:], f"time for PC {PC_num+1}", None
        elif dpca_axis == "stimulus":
            y, label, axes_to_loop = Z['s'][PC_num, :], f"stimulus PC {PC_num+1}", 's' if axes_to_loop is None else axes_to_loop
        elif dpca_axis == 'choice':
            y, label, axes_to_loop = Z['c'][PC_num, :], f"choice PC {PC_num+1}", 'c' if axes_to_loop is None else axes_to_loop
        if flip:
            y = -y
        plot_component(ax[idx], x, y, ylim=ylim, plot_mean=True, axes_to_loop=axes_to_loop, color=color,lw=4)
        # else:
        #     plot_component(ax[idx], x, y, plot_mean=True, axes_to_loop=axes_to_loop, color=color,lw=lw)
        ax[idx].set_xlabel(alignment.capitalize().replace('_', ' '), fontsize=15)
        ax[idx].spines['right'].set_visible(False)
        ax[idx].spines['top'].set_visible(False)

    # ax[1].set_yticks([])
    # ax[1].spines['left'].set_visible(False)
    # ax[0].set_ylabel(f"Projection onto {dpca_axis.capitalize()} PC {PC_num+1} (a.u.)", fontsize=15)

    if legend:
        # Add the legend for different lines
        shade_colors = ['#b3b3b3', '#7f7f7f', '#4d4d4d', '#000000']
        handles = [
                    plt.Line2D([0], [0], color=shade_colors[3], label="50% coh"),
                    plt.Line2D([0], [0], color=shade_colors[2], label="20% coh"),
                    plt.Line2D([0], [0], color=shade_colors[1], label="6% coh"),
                    plt.Line2D([0], [0], color=shade_colors[0], label="0% coh"),
                    plt.Line2D([0], [0], color=plot_params["color"][3][0], label="toRF choice"),
                    plt.Line2D([0], [0], color=plot_params["color"][3][1], label="awayRF choice"),
                ]
        fig.legend(handles=handles, bbox_to_anchor=(1, 1), loc='upper right')

    ax[0].set_title(f"Projection of {title} {dpca_axis.capitalize()} Component Across Task Epochs")
    plt.show()

In [ ]:
dpca_axis = "stimulus"

# dpca_results = visual_projection
# fig, ax = plt.subplots(3,1, figsize = (5,15))
# plot_single_pc_projection(ax,dpca_axis=dpca_axis, prior="toRF_prior", dpca_results=dpca_results, legend=False, lw=2.5,title='Visual Epoch')

# dpca_results = response_projection
# fig, ax = plt.subplots(3,1, figsize = (5,15))
# plot_single_pc_projection(ax,dpca_axis=dpca_axis, prior="toRF_prior", dpca_results=dpca_results, legend=False,lw=2.5,title='Response Epoch')

dpca_results = response_projection
fig, ax = plt.subplots(3,1, figsize = (5,15))
plot_single_pc_projection(ax,dpca_axis=dpca_axis, dpca_results=dpca_results, legend=True,lw=2.5,title='Cue Epoch',axes_to_loop=['s','c'], flip=True)

In [ ]:
dpca_axis = "choice"

# dpca_results = visual_projection
# fig, ax = plt.subplots(3,1, figsize = (5,15))
# plot_single_pc_projection(ax,dpca_axis=dpca_axis, prior="toRF_prior", dpca_results=dpca_results, legend=False, lw=2.5,title='Visual Epoch')

# dpca_results = response_projection
# fig, ax = plt.subplots(3,1, figsize = (5,15))
# plot_single_pc_projection(ax,dpca_axis=dpca_axis, prior="toRF_prior", dpca_results=dpca_results, legend=False,lw=2.5,title='Response Epoch')

dpca_results = cue_projection
fig, ax = plt.subplots(3,1, figsize = (5,15))
plot_single_pc_projection(ax,dpca_axis=dpca_axis, dpca_results=dpca_results, legend=False,lw=2.5,title='Cue Epoch')

#### Neuron Selectivity

In [ ]:
selectivity_align_labels = align_labels.copy()

In [ ]:
loadings = {}
for dpca_axis in ['c', 's', 't']:
    loadings[dpca_axis] = []
    for alignment in selectivity_align_labels:
        loadings[dpca_axis].append(dpca_projection[alignment]["model"].D[dpca_axis][:,0])
    loadings[dpca_axis] = np.array(loadings[dpca_axis])


In [ ]:
def plot_selectiveity_summary(loadings, total_neurons,threshold=0.3):
    # Prepare summary counts for top row plots
    summary_list = []

    for idx, align_label in enumerate(selectivity_align_labels):
        selective_neurons = 0
        mixed_neurons = 0
        nonselective_neurons = 0

        for neuron in range(total_neurons):
            vals = np.array([np.abs(loadings[axis][idx][neuron]) for axis in ['c', 's']])
            strong_axes_count = np.sum(vals > threshold)
            if strong_axes_count == 0:
                nonselective_neurons += 1
            elif strong_axes_count == 1:
                selective_neurons += 1
            else:
                mixed_neurons += 1

        summary_list.append({
            'Selective': selective_neurons,
            'Mixed-selective': mixed_neurons,
            'Non-selective': nonselective_neurons
        })

    # Precompute venn data for bottom row (only c and s)
    venn_data_list = []

    for idx, align_label in enumerate(selectivity_align_labels):
        c_set = set(np.where(np.abs(loadings['c'][idx]) > threshold)[0])
        s_set = set(np.where(np.abs(loadings['s'][idx]) > threshold)[0])

        venn_data = {
            '10': len(c_set - s_set),         # only c
            '01': len(s_set - c_set),         # only s
            '11': len(c_set & s_set),         # both
            'nonselective': len(set(range(total_neurons)) - (c_set | s_set))
        }
        venn_data_list.append(venn_data)

    # Create figure with 2 rows, 3 columns
    fig, axes = plt.subplots(2, 4, figsize=(24, 10), gridspec_kw={'height_ratios': [1, 1.5]})

    fontsize = 15

    # Top row: pie charts unchanged
    for idx, ax in enumerate(axes[0]):
        summary = summary_list[idx]
        labels = ['Selective', 'Mixed-selective', 'Non-selective']
        counts = [summary[label] for label in labels]
        colors = ['royalblue', 'tomato', 'gray']

        wedges, texts, autotexts = ax.pie(
            counts, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90, counterclock=False,
            textprops={'fontsize': fontsize}
        )
        for text in texts:
            text.set_fontsize(fontsize)
        for autotext in autotexts:
            autotext.set_fontsize(fontsize)

        ax.set_title(
            f"{list(selectivity_align_labels.keys())[idx].replace('_',' ').capitalize()}",
            fontsize=fontsize+6, pad=25
        )
        ax.axis('equal')

    # Bottom row: venn2 only (Choice vs Stimulus)
    for idx, ax in enumerate(axes[1]):
        data = venn_data_list[idx]

        subset_sizes = (data['10'], data['01'], data['11'])

        v = venn2(subsets=subset_sizes, set_labels=('Choice', 'Stimulus'), ax=ax)

        # subset label text: convert to %
        for subset_id in ('10', '01', '11'):
            label = v.get_label_by_id(subset_id)
            if label:
                count = data[subset_id]
                label.set_text(f"{100 * count / total_neurons:.1f}%")
                label.set_fontsize(fontsize)

        # set label fonts
        for set_label in v.set_labels:
            if set_label:
                set_label.set_fontsize(fontsize)

        ax.set_aspect('equal')

    plt.tight_layout()
    plt.show()


In [ ]:
total_neurons = loadings['c'].shape[1]
plot_selectiveity_summary(loadings, total_neurons, threshold=0.3)

### loading heatmap

#### cosine similarity of demixed PCs

In [ ]:
def pairwise_cosine_similarity(vecs):
    n = len(vecs)
    sim = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            sim[i, j] = 1 - cosine(vecs[i], vecs[j])
    return sim

similarity_matrix = {"toRF": [], "awayRF": []}

loadings = awayRF_loadings
fig, axs = plt.subplots(1, 3, figsize=(20, 6))
dpca_axes = ['b', 's', 'c']

for idx, dpca_ax in enumerate(dpca_axes):
    similarity_matrix["awayRF"].append(pairwise_cosine_similarity(loadings[dpca_ax]))
    # Mask lower triangle and diagonal
    mask = np.triu(np.ones_like(similarity_matrix["awayRF"][idx], dtype=bool), k=1)
    sns.heatmap(similarity_matrix["awayRF"][idx],
                annot=True,
                annot_kws={"size": 14},
                xticklabels=[align_label.split('_')[0].capitalize() for align_label in align_labels],
                yticklabels=[align_label.split('_')[0].capitalize() for align_label in align_labels],
                cmap='coolwarm',
                vmin=-1,
                vmax=1,
                square=True,
                mask=mask,
                cbar=(idx == len(dpca_axes) - 1),  # only last subplot has colorbar
                ax=axs[idx])

    axs[idx].tick_params(axis='x', labelsize=12)
    axs[idx].tick_params(axis='y', labelsize=12)
    axs[idx].set_title(f"{marginalization[dpca_ax]} Axis",
                       fontsize=16, pad=20)
axs[0].set_ylabel("Aligned to", fontsize=16, labelpad=15)
axs[1].set_xlabel("Aligned to", fontsize=16, labelpad=15)

plt.suptitle("Cosine Similarity Across Task Epochs", fontsize=22)
plt.subplots_adjust(top=0.80, wspace=0.15)
# plt.tight_layout()
plt.show()

loadings = toRF_loadings
fig, axs = plt.subplots(1, 3, figsize=(20, 6))
dpca_axes = ['b', 's', 'c']
for idx, dpca_ax in enumerate(dpca_axes):
    similarity_matrix["toRF"].append(pairwise_cosine_similarity(loadings[dpca_ax]))
    # Mask lower triangle and diagonal
    mask = np.triu(np.ones_like(similarity_matrix["toRF"][idx], dtype=bool), k=1)
    sns.heatmap(similarity_matrix["toRF"][idx],
                annot=True,
                annot_kws={"size": 14},
                xticklabels=[align_label.split('_')[0].capitalize() for align_label in align_labels],
                yticklabels=[align_label.split('_')[0].capitalize() for align_label in align_labels],
                cmap='coolwarm',
                vmin=-1,
                vmax=1,
                square=True,
                mask=mask,
                cbar=(idx == len(dpca_axes) - 1),  # only last subplot has colorbar
                ax=axs[idx])

    axs[idx].tick_params(axis='x', labelsize=12)
    axs[idx].tick_params(axis='y', labelsize=12)
    axs[idx].set_title(f"{marginalization[dpca_ax]} Axis",
                       fontsize=16, pad=20)
axs[0].set_ylabel("Aligned to", fontsize=16, labelpad=15)
axs[1].set_xlabel("Aligned to", fontsize=16, labelpad=15)

plt.suptitle("Cosine Similarity Across Task Epochs", fontsize=22)
plt.subplots_adjust(top=0.80, wspace=0.15)
# plt.tight_layout()
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

dpca_axes = ['b', 's', 'c']
corr_dict = {"toRF": [], "awayRF": []}

loadings = awayRF_loadings
fig, axs = plt.subplots(1, 3, figsize=(20, 6))
# First, compute all correlation matrices to find global vmin and vmax
all_corrs = []
for idx in range(len(align_labels)):
    corr_matrix = np.zeros((len(dpca_axes), len(dpca_axes)))
    for i, a1 in enumerate(dpca_axes):
        for j, a2 in enumerate(dpca_axes):
            corr_matrix[i, j] = 1 - cosine(loadings[a1][idx], loadings[a2][idx])
    all_corrs.append(corr_matrix)

# # Find global min and max for color scale

for idx, align_label in enumerate(align_labels):
    corr_matrix = all_corrs[idx]

    # Mask upper triangle (excluding diagonal)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

    sns.heatmap(corr_matrix,
                xticklabels=[marginalization[k] for k in dpca_axes],
                yticklabels=[marginalization[k] for k in dpca_axes],
                annot=True,
                annot_kws={"size": 14},   # <-- increase annotation font size here
                cmap="coolwarm",
                mask=mask,
                square=True,
                vmin=-1,
                vmax=1,
                cbar=(idx == len(align_labels) - 1),  # show colorbar only on last plot
                ax=axs[idx])
    axs[idx].tick_params(axis='x', labelsize=14)
    axs[idx].tick_params(axis='y', labelsize=14)
    axs[idx].set_title(f"Aligned to: {align_label.replace('_', ' ').capitalize()}", fontsize=16, pad=20)

corr_dict["awayRF"] = np.array(all_corrs)
plt.suptitle("Cosine Similarity Between dPCA Axes", fontsize=22)
plt.subplots_adjust(top=0.80, wspace=0.15)
# plt.tight_layout()
plt.show()


loadings = toRF_loadings
fig, axs = plt.subplots(1, 3, figsize=(20, 6))
# First, compute all correlation matrices to find global vmin and vmax
all_corrs = []
for idx in range(len(align_labels)):
    corr_matrix = np.zeros((len(dpca_axes), len(dpca_axes)))
    for i, a1 in enumerate(dpca_axes):
        for j, a2 in enumerate(dpca_axes):
            corr_matrix[i, j] = 1 - cosine(loadings[a1][idx], loadings[a2][idx])
    all_corrs.append(corr_matrix)

# # Find global min and max for color scale

for idx, align_label in enumerate(align_labels):
    corr_matrix = all_corrs[idx]

    # Mask upper triangle (excluding diagonal)
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

    sns.heatmap(corr_matrix,
                xticklabels=[marginalization[k] for k in dpca_axes],
                yticklabels=[marginalization[k] for k in dpca_axes],
                annot=True,
                annot_kws={"size": 14},   # <-- increase annotation font size here
                cmap="coolwarm",
                mask=mask,
                square=True,
                vmin=-1,
                vmax=1,
                cbar=(idx == len(align_labels) - 1),  # show colorbar only on last plot
                ax=axs[idx])
    axs[idx].tick_params(axis='x', labelsize=14)
    axs[idx].tick_params(axis='y', labelsize=14)
    axs[idx].set_title(f"Aligned to: {align_label.replace('_', ' ').capitalize()}", fontsize=16, pad=20)

corr_dict["toRF"] = np.array(all_corrs)
plt.suptitle("Cosine Similarity Between dPCA Axes", fontsize=22)
plt.subplots_adjust(top=0.80, wspace=0.15)
# plt.tight_layout()
plt.show()


In [ ]:
choice_bias_idx = [2,0]
stimulus_bias_idx = [1,0]
stimulus_choice_idx = [1,2]
plt.figure(figsize=(16, 6))
plt.subplot(1, 3, 1)
plt.plot(corr_dict['toRF'][:,choice_bias_idx[0],choice_bias_idx[1]], label='toRF prior', color='blue', marker='o', linewidth=3, markersize=8)
plt.plot(corr_dict['awayRF'][:,choice_bias_idx[0],choice_bias_idx[1]], label='awayRF prior', color='red', marker='o', linewidth=3, markersize=8)
plt.xlim(-0.2, 2.2)
plt.ylim(-1, 1)
plt.title('Choice-Bias Axes', fontsize=18)
plt.xticks([0,1,2], [align_label.replace('_', ' ').capitalize() for align_label in align_labels], fontsize=14)
plt.yticks(np.arange(-1, 1.1, 0.5), fontsize=14)
plt.subplot(1, 3, 2)
plt.plot([1,2], corr_dict['toRF'][1:,stimulus_choice_idx[0],stimulus_choice_idx[1]], label='toRF prior', color='blue', marker='o', linewidth=3, markersize=8)
plt.plot([1,2], corr_dict['awayRF'][1:,stimulus_choice_idx[0],stimulus_choice_idx[1]], label='awayRF prior', color='red', marker='o', linewidth=3, markersize=8)
plt.ylim(-1, 1)
plt.title('Stimulus-Choice Axes', fontsize=18)
plt.xticks([0,1,2], [align_label.replace('_', ' ').capitalize() for align_label in align_labels], fontsize=14)
plt.yticks(np.arange(-1, 1.1, 0.5), fontsize=14)
plt.xlim(0.8, 2.2)
plt.subplot(1, 3, 3)
plt.plot([1,2], corr_dict['toRF'][1:,stimulus_bias_idx[0],stimulus_bias_idx[1]], label='toRF prior', color='blue', marker='o', linewidth=3, markersize=8)
plt.plot([1,2], corr_dict['awayRF'][1:,stimulus_bias_idx[0],stimulus_bias_idx[1]], label='awayRF prior', color='red', marker='o', linewidth=3, markersize=8)
plt.ylim(-1, 1)
plt.title('Stimulus-Bias Axes', fontsize=18)
plt.xticks([0,1,2], [align_label.replace('_', ' ').capitalize() for align_label in align_labels], fontsize=14)
plt.yticks(np.arange(-1, 1.1, 0.5), fontsize=14)
plt.xlim(0.8, 2.2)

plt.suptitle('Cosine Similarity Between dPCA Axes', fontsize=22)



plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
choice_bias_idx = [2,0]
plt.figure(figsize=(8, 6))
plt.plot(corr_dict['toRF'][:,choice_bias_idx[0],choice_bias_idx[1]], label='toRF prior', color='blue', marker='o', linewidth=3, markersize=8)
plt.plot(corr_dict['awayRF'][:,choice_bias_idx[0],choice_bias_idx[1]], label='awayRF prior', color='red', marker='o', linewidth=3, markersize=8)
plt.xlim(-0.2, 2.2)
plt.ylim(-1, 1)
plt.title('Cosine Similarity Between Choice-Bias Axes Across Alignments', fontsize=18)
plt.xticks([0,1,2], [align_label.replace('_', ' ').capitalize() for align_label in align_labels], fontsize=14)
plt.yticks(np.arange(-1, 1.1, 0.5), fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(loadings['c'][i], loadings['b'][i])
plt.xlabel("Choice Loadings")
plt.ylabel("Bias Loadings")
plt.title(f"Neuron-wise Comparison (alignment {i})")
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.grid(True)
plt.show()


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

X = []
labels = []

for dpca_axis in loadings:
    for i, align in enumerate(align_labels):
        vec = loadings[dpca_axis][i]
        X.append(vec)
        labels.append(f"{dpca_axis}_{align}")

X = np.array(X)
proj = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(8,6))
for label in set(labels):
    idxs = [i for i, l in enumerate(labels) if l == label]
    plt.scatter(proj[idxs, 0], proj[idxs, 1], label=label)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.title("PCA of dPCA Loadings (Component × Alignment)")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

for dpca_axis in ['b', 'c', 's']:
    var_across_align = np.var(loadings[dpca_axis], axis=0)
    top_changing = np.argsort(var_across_align)[::-1][:10]

    plt.bar(range(10), var_across_align[top_changing])
    plt.title(f"Top Changing Neurons for Axis '{dpca_axis}'")
    plt.xlabel("Neuron Rank")
    plt.ylabel("Variance Across Alignments")
    plt.show()


#### loading mapping

In [ ]:
raw_loadings = []
for axis_idx, dpca_axis in enumerate(['c', 's', 't']):
    loadings_axis = []
    for alignment in ['cue','response']:
        loadings_axis.append(dpca_projection[alignment]["model"].D[dpca_axis][:,0])
    raw_loadings.append(np.array(loadings_axis))
raw_loadings = np.vstack(raw_loadings).T


In [ ]:
# quantile = 0.005
# quantile_loadings = np.clip(raw_loadings, np.quantile(raw_loadings, quantile), np.quantile(raw_loadings, 1-quantile))
# sorted_idx = np.argsort(quantile_loadings[:,-1])
# sns.heatmap(quantile_loadings[sorted_idx,:])


In [ ]:
sorted_idx = np.argsort(raw_loadings[:,-1])
sns.heatmap(raw_loadings[sorted_idx,:])

In [ ]:
dpca_axis_mapping = {
    "coherence": 's',
    "choice": 'c',
    # "choice-stimulus": 'sc',
    # "time": 't'
}

In [ ]:
# flip loading for some axis, so that positive values indicate higher firing for high coherence and toRF choice
flip_axes = {'s':{'cue': False, 'response': True}, 'c':{'cue': True, 'response': True}}

loadings = {}
epochs = ['cue','response']
for dpca_axis in ['c', 's']:
    loadings[dpca_axis] = []
    for alignment in epochs:
        if flip_axes[dpca_axis][alignment]:
            loadings[dpca_axis].append(-dpca_projection[alignment]["model"].D[dpca_axis][:,0])
        else:
            loadings[dpca_axis].append(dpca_projection[alignment]["model"].D[dpca_axis][:,0])
    loadings[dpca_axis] = np.array(loadings[dpca_axis])
# for dpca_axis in ['c', 's', 't']:
#     loadings[dpca_axis] = np.abs(loadings[dpca_axis])


In [ ]:
# sort by cell type and within each cell type by the time axis loading in the last epoch
neuron_types = ['visual_phasic', 'visual_tonic', 'visual_motor', 'buildup','motor','unknown']
sorted_type_indices = []
type_bound = np.zeros(len(neuron_types)+1)
for i, type in enumerate(neuron_types):
    type_indices = neuron_metadata.index[neuron_metadata['classification'] == type].tolist()
    sorted_type_indices.append(np.array(type_indices)[np.argsort(loadings['c'][-2, type_indices])])
    type_bound[i+1] = type_bound[i] + len(type_indices)


#### clustering of dpca loadings

In [ ]:
import plotly.graph_objects as go

alignment = 'response'

# Extract the dPCA components
x = dpca_projection[alignment]["model"].D['s'][:, 0]
y = dpca_projection[alignment]["model"].D['s'][:, 1]
z = dpca_projection[alignment]["model"].D['c'][:, 1]

# Create interactive 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers',
    marker=dict(
        size=5,
        color='black',  # all points same color
        opacity=0.8
    ),
    text=[f'Point {i}' for i in range(len(x))],  # optional hover text
    hoverinfo='text'
)])

fig.update_layout(
    scene=dict(
        xaxis_title='coherence PC 1',
        yaxis_title='coherence PC 2',
        zaxis_title='choice PC 2'
    ),
    title='3D Scatter dPCA Projection'
)

fig.show()


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
import plotly.graph_objects as go

alignment = 'response'

# Extract the dPCA components
x = dpca_projection[alignment]["model"].D['s'][:, 0]
y = dpca_projection[alignment]["model"].D['s'][:, 1]
z = dpca_projection[alignment]["model"].D['c'][:, 1]

# Stack into 3D points for clustering
points = np.stack([x, y, z], axis=1)

# Perform k-means clustering
n_clusters = 10  # choose number of clusters
kmeans = KMeans(n_clusters=n_clusters, random_state=0)
labels = kmeans.fit_predict(points)

# Create interactive 3D scatter plot with cluster colors
fig = go.Figure(data=[go.Scatter3d(
    x=x,
    y=y,
    z=z,
    mode='markers',
    marker=dict(
        size=5,
        color=labels,       # color by cluster
        colorscale='Viridis',
        opacity=0.8
    ),
    text=[f'Point {i}, Cluster {labels[i]}' for i in range(len(x))],
    hoverinfo='text'
)])

fig.update_layout(
    scene=dict(
        xaxis_title='coherence PC 1',
        yaxis_title='coherence PC 2',
        zaxis_title='choice PC 2'
    ),
    title='3D Scatter dPCA Projection with Clusters'
)

fig.show()


#### cell type classified, compare choice and coherence in same epoch

In [ ]:
colormap = 'jet'

In [ ]:
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

n_epochs = len(epochs)
n_axes = len(dpca_axis_mapping)  # e.g., t, s, c


# ---- Create figure with gridspec ----
# width ratios: axes inside each epoch tightly packed + spacer column at the end
width_ratios = []
for _ in range(n_epochs):
    width_ratios.extend([1]*n_axes)  # axes inside epoch
    width_ratios.append(0.1)         # spacer between epochs
width_ratios[-1] = 0.1              # last column is colorbar

fig = plt.figure(figsize=(4*n_epochs, 10))
gs = gridspec.GridSpec(1, n_epochs*(n_axes+1), width_ratios=width_ratios, wspace=0.0)

# ---- Keep track of all axes ----
all_axes = []

# ---- Plot heatmaps ----
for epoch_idx, alignment in enumerate(epochs):

    # ---- Compute global vmin/vmax in each epochs across axes ----
    all_values = []
    
    for label, dpca_axis in dpca_axis_mapping.items():
        vals = loadings[dpca_axis][epoch_idx, :]
        all_values.append(vals)
    all_values = np.array(all_values)
    # vmin, vmax = np.min(all_values), np.max(all_values)
    std = np.std(all_values)


    epoch_axes = []  # store axes of this epoch
    for ax_idx, (label, dpca_axis) in enumerate(dpca_axis_mapping.items()):
        subplot_idx = epoch_idx*(n_axes+1) + ax_idx
        ax = fig.add_subplot(gs[0, subplot_idx])
        epoch_axes.append(ax)
        all_axes.append(ax)
        

        # Concatenate data from each cell type, inserting NaN rows as white separators
        data_segments = []
        for i, idx in enumerate(sorted_type_indices):
            segment = loadings[dpca_axis][epoch_idx, idx][:, np.newaxis]
            data_segments.append(segment)
            if i < len(sorted_type_indices) - 1:
                gap = np.full((3, 1), np.nan)  # 3-row white gap; adjust as needed
                data_segments.append(gap)
        data = np.vstack(data_segments)
        data = np.clip(data/std,-2,2)
        
        sns.heatmap(
            data,
            cmap=colormap,
            ax=ax,
            cbar=False,
            # norm= norm,
        )
        ax.set_xticks([0.5])
        ax.set_xticklabels([f"{label.capitalize()} PC 1"], fontsize=14,y=-0.01)
        ax.set_yticks([])


    # Add epoch title above the first axis of this epoch
    epoch_axes[1].set_title(f"{alignment.capitalize()} Epoch", fontsize=16, pad=15,x=0)
for i, type in enumerate(neuron_types):
    all_axes[0].text(-0.6, (type_bound[i]+type_bound[i+1])/2+2*i+1, type.replace('_', ' ').title(), ha='center', va='center', fontsize=16)
# ---- Single shared colorbar on the far right ----
cax = fig.add_subplot(gs[0, -1])

sm = plt.cm.ScalarMappable(cmap='jet', norm=plt.Normalize(vmin=-2, vmax=2))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation='vertical')
cbar.set_label('Normalized Loading', fontsize=14)

# ---- Layout ----
fig.subplots_adjust(top=0.9, bottom=0.05, left=0.05, right=0.95)
# fig.suptitle('Axis Loadings Across Epochs', fontsize=16)
plt.show()



#### compare choice and coherence in same epoch

In [ ]:
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

n_epochs = len(epochs)
n_axes = len(dpca_axis_mapping)  # e.g., t, s, c
colormap = 'jet'

sorted_indices = np.argsort(loadings['c'][-2,:])

# ---- Create figure with gridspec ----
# width ratios: axes inside each epoch tightly packed + spacer column at the end
width_ratios = []
for _ in range(n_epochs):
    width_ratios.extend([1]*n_axes)  # axes inside epoch
    width_ratios.append(0.1)         # spacer between epochs
width_ratios[-1] = 0.1              # last column is colorbar

fig = plt.figure(figsize=(4*n_epochs, 10))
gs = gridspec.GridSpec(1, n_epochs*(n_axes+1), width_ratios=width_ratios, wspace=0.0)

# ---- Keep track of all axes ----
all_axes = []

# ---- Plot heatmaps ----
for epoch_idx, alignment in enumerate(epochs):

    # ---- Compute global vmin/vmax in each epochs across axes ----
    all_values = []
    
    for label, dpca_axis in dpca_axis_mapping.items():
        vals = loadings[dpca_axis][epoch_idx, :]
        all_values.append(vals)
    all_values = np.array(all_values)
    vmin, vmax = np.min(all_values), np.max(all_values)
    
    epoch_axes = []  # store axes of this epoch
    for ax_idx, (label, dpca_axis) in enumerate(dpca_axis_mapping.items()):
        subplot_idx = epoch_idx*(n_axes+1) + ax_idx
        ax = fig.add_subplot(gs[0, subplot_idx])
        epoch_axes.append(ax)
        all_axes.append(ax)
        
        data = loadings[dpca_axis][epoch_idx, sorted_indices][:, np.newaxis]
        data = 2*(data - vmin) / (vmax - vmin) -1 # normalize 0-1 across all epochs
        
        sns.heatmap(
            data,
            cmap=colormap,
            ax=ax,
            cbar=False
        )
        ax.set_xticks([0.5])
        ax.set_xticklabels([f"{label.capitalize()} PC 1"], fontsize=14,y=-0.01)
        ax.set_yticks([])

    # Add epoch title above the first axis of this epoch
    epoch_axes[1].set_title(f"{alignment.capitalize()} Epoch", fontsize=16, pad=15,x=0)

# ---- Single shared colorbar on the far right ----
cax = fig.add_subplot(gs[0, -1])
sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation='vertical')
cbar.set_label('Normalized Loading', fontsize=14)  # set font size here

# ---- Layout ----
fig.subplots_adjust(top=0.9, bottom=0.05, left=0.05, right=0.95)
# fig.suptitle('Axis Loadings Across Epochs', fontsize=16)
plt.show()



#### compare same demixed pc in different epoch

In [ ]:
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

n_epochs = len(epochs)
n_axes = len(dpca_axis_mapping)  # e.g., t, s, c
colormap = 'jet'

sorted_indices = np.argsort(loadings['c'][-2,:])

# ---- Create figure with gridspec ----
# width ratios: axes inside each epoch tightly packed + spacer column at the end
width_ratios = []
for _ in range(n_axes):
    width_ratios.extend([1]*n_epochs)  # axes inside epoch
    width_ratios.append(0.1)         # spacer between epochs
width_ratios[-1] = 0.1              # last column is colorbar

fig = plt.figure(figsize=(4*n_epochs, 10))
gs = gridspec.GridSpec(1, (n_epochs+1)*(n_axes), width_ratios=width_ratios, wspace=0.0)

# ---- Keep track of all axes ----
all_axes = []

# ---- Plot heatmaps ----
for epoch_idx, alignment in enumerate(epochs):

    # ---- Compute global vmin/vmax in each epochs across axes ----
    all_values = []
    
    for label, dpca_axis in dpca_axis_mapping.items():
        vals = loadings[dpca_axis][epoch_idx, :]
        all_values.append(vals)
    all_values = np.array(all_values)
    vmin, vmax = np.min(all_values), np.max(all_values)
    
    epoch_axes = []  # store axes of this epoch
    for ax_idx, (label, dpca_axis) in enumerate(dpca_axis_mapping.items()):
        subplot_idx = epoch_idx + ax_idx * (n_epochs + 1)
        ax = fig.add_subplot(gs[0, subplot_idx])
        epoch_axes.append(ax)
        all_axes.append(ax)
        
        data = loadings[dpca_axis][epoch_idx, sorted_indices][:, np.newaxis]
        data = 2*(data - vmin) / (vmax - vmin) -1 # normalize 0-1 across all epochs
        
        sns.heatmap(
            data,
            cmap=colormap,
            ax=ax,
            cbar=False
        )
        ax.set_xticks([0.5])
        ax.set_xticklabels([f"{label.capitalize()} PC 1"], fontsize=14,y=-0.01)
        ax.set_yticks([])

    # Add epoch title above the first axis of this epoch
    epoch_axes[1].set_title(f"{alignment.capitalize()} Epoch", fontsize=14, pad=15)
    epoch_axes[0].set_title(f"{alignment.capitalize()} Epoch", fontsize=14, pad=15)

# ---- Single shared colorbar on the far right ----
cax = fig.add_subplot(gs[0, -1])
sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax, orientation='vertical')
cbar.set_label('Normalized Loading', fontsize=14)  # set font size here

# ---- Layout ----
fig.subplots_adjust(top=0.9, bottom=0.05, left=0.05, right=0.95)
# fig.suptitle('Axis Loadings Across Epochs', fontsize=16)
plt.show()



#### cell type classified, compare same demixed pc in different epoch

In [ ]:
import matplotlib.gridspec as gridspec
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

n_epochs = len(epochs)
n_axes = len(dpca_axis_mapping)  # e.g., t, s, c
colormap = 'jet'

# ---- Create figure with gridspec ----
# width ratios: axes inside each epoch tightly packed + spacer column at the end
width_ratios = []
for _ in range(n_axes):
    width_ratios.extend([1]*n_epochs)  # axes inside epoch
    width_ratios.append(0.1)         # spacer between epochs
width_ratios[-1] = 0.1              # last column is colorbar

fig = plt.figure(figsize=(4*n_epochs, 10))
gs = gridspec.GridSpec(1, (n_epochs + 1)*(n_axes), width_ratios=width_ratios, wspace=0.0)

# ---- Keep track of all axes ----
all_axes = []

# ---- Plot heatmaps ----
for epoch_idx, alignment in enumerate(epochs):

    # ---- Compute global vmin/vmax in each epochs across axes ----
    all_values = []
    
    for label, dpca_axis in dpca_axis_mapping.items():
        vals = loadings[dpca_axis][epoch_idx, :]
        all_values.append(vals)
    all_values = np.array(all_values)
    vmin, vmax = np.min(all_values), np.max(all_values)
    
    epoch_axes = []  # store axes of this epoch
    for ax_idx, (label, dpca_axis) in enumerate(dpca_axis_mapping.items()):
        subplot_idx =  ax_idx * (n_epochs + 1) + epoch_idx
        ax = fig.add_subplot(gs[0, subplot_idx])
        epoch_axes.append(ax)
        all_axes.append(ax)
        
        # Concatenate data from each cell type, inserting NaN rows as white separators
        data_segments = []
        for i, idx in enumerate(sorted_type_indices):
            segment = loadings[dpca_axis][epoch_idx, idx][:, np.newaxis]
            data_segments.append(segment)
            if i < len(sorted_type_indices) - 1:
                gap = np.full((1, 1), np.nan)  # 3-row white gap; adjust as needed
                data_segments.append(gap)
        data = np.vstack(data_segments)
        data = 2 * (data - vmin) / (vmax - vmin) - 1 # normalize 0-1 across all epochs
        
        sns.heatmap(
            data,
            cmap=colormap,
            ax=ax,
            cbar=False
        )
        ax.set_xticks([0.5])
        ax.set_xticklabels([f"{label.capitalize()} PC 1"], fontsize=14)
        ax.set_yticks([])

    # Add epoch title above the first axis of this epoch
    epoch_axes[0].set_title(f"{alignment} epoch", fontsize=14, pad=15)
    epoch_axes[1].set_title(f"{alignment} epoch", fontsize=14, pad=15)
for i, type in enumerate(neuron_types):
    all_axes[0].text(-0.5, (type_bound[i]+type_bound[i+1])/2+2*i+1, type, ha='center', va='center', fontsize=16)
# ---- Single shared colorbar on the far right ----
cax = fig.add_subplot(gs[0, -1])
sm = plt.cm.ScalarMappable(cmap=colormap, norm=plt.Normalize(vmin=-1, vmax=1))
sm.set_array([])
fig.colorbar(sm, cax=cax, orientation='vertical', label='Normalized Loading')

# ---- Layout ----
fig.subplots_adjust(top=0.9, bottom=0.05, left=0.05, right=0.95)
# fig.suptitle('Axis Loadings Across Epochs', fontsize=16)
plt.show()

